<a href="https://colab.research.google.com/github/ParusSlava/DTA_2026/blob/main/ML/logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [1]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [5]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [26]:
# TODO: виведи баланс класів
clients["upgraded"].value_counts(normalize=True).round(3)

clients.dtypes

,0
age,int64
tenure,int64
usage,int64
support,int64
plan,object
region,object
upgraded,int64


✍️ Випиши списки стовпців (знадобляться далі):
> числові: 'age', 'tenure', 'usage', 'support','upgraded'    
> категорійні: 'plan', 'region'

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [15]:
from sklearn.model_selection import train_test_split

# TODO: створи X, y та зроби поділ
X = clients[['age', 'tenure', 'usage', 'support', 'plan', 'region' ]]
y = clients["upgraded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Баланс у train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Баланс у test: ", y_test.value_counts(normalize=True).round(3).to_dict())


X_train: (720, 6)  X_test: (180, 6)
Баланс у train: {0: 0.515, 1: 0.485}
Баланс у test:  {0: 0.517, 1: 0.483}


### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: задай num_cols, cat_cols і збери preprocess

num_cols = ["age", "tenure", "usage", "support"]
cat_cols = ["plan", "region"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TODO: збери pipe

pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

print(pipe)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'tenure', 'usage',
                                                   'support']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plan', 'region'])])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])


### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [19]:
# TODO: навчи pipe і виведи accuracy на тесті

pipe.fit(X_train, y_train)

acc = pipe.score(X_test, y_test)
print(f"Accuracy на тесті: {acc:.2%}")

Accuracy на тесті: 84.44%


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [30]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = pipe.predict(X_test)

cm = pd.DataFrame(confusion_matrix(y_test, y_pred),
                  index=["Насправді: ні", "Насправді: так"],
                  columns=["Прогноз: ні", "Прогноз: так"])

print("=== Confusion Matrix ===")
print(cm)

print()
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["не преміум", "преміум"]))

=== Confusion Matrix ===
                Прогноз: ні  Прогноз: так
Насправді: ні            80            13
Насправді: так           15            72

=== Classification Report ===
              precision    recall  f1-score   support

  не преміум       0.84      0.86      0.85        93
     преміум       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [31]:
from sklearn.metrics import roc_auc_score

# TODO: дістань proba та порахуй ROC-AUC
proba = pipe.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, proba)
print(f"ROC-AUC: {auc:.3f}")

print()
print("Перші 5 ймовірностей:")
for i, p in enumerate(proba[:5]):
    print(f"  клієнт {i}: P(преміум) = {p:.2%}")

ROC-AUC: 0.927

Перші 5 ймовірностей:
  клієнт 0: P(преміум) = 6.77%
  клієнт 1: P(преміум) = 15.17%
  клієнт 2: P(преміум) = 0.17%
  клієнт 3: P(преміум) = 96.89%
  клієнт 4: P(преміум) = 18.74%


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [32]:
# TODO: побудуй таблицю "ознака — коефіцієнт", відсортовану за |коеф.|

# Витягуємо назви ознак після препроцесингу
num_names = num_cols
cat_names = (pipe
    .named_steps["prep"]
    .named_transformers_["cat"]
    .get_feature_names_out(cat_cols)
    .tolist()
)
all_names = num_names + cat_names

# Коефіцієнти моделі
coefs = pipe.named_steps["model"].coef_[0]

# Зводимо у DataFrame
coef_df = pd.DataFrame({
    "ознака": all_names,
    "коефіцієнт": coefs
})
coef_df["|коефіцієнт|"] = coef_df["коефіцієнт"].abs()
coef_df = coef_df.sort_values("|коефіцієнт|", ascending=False).reset_index(drop=True)

print(coef_df.to_string(index=False))

        ознака  коефіцієнт  |коефіцієнт|
         usage    2.027135      2.027135
        tenure    1.648386      1.648386
 plan_сімейний    1.411953      1.411953
  plan_базовий   -1.339728      1.339728
       support   -0.835947      0.835947
  region_захід    0.537637      0.537637
           age   -0.406875      0.406875
   region_схід   -0.340620      0.340620
region_південь   -0.189350      0.189350
 region_північ    0.136239      0.136239
 plan_стандарт    0.071680      0.071680


✍️ **Відповідь словами:**
> Найсильніше підвищує шанс переходу на преміум usage (кількість годин використання), а знижує plan_базовий (базовий тариф). Клієнти які багато користуються сервісом і давно з ним — найкращі кандидати на преміум. А ті хто рідко користується і часто звертається у підтримку — навпаки..

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [33]:
# TODO: створи new_client, виведи рішення та ймовірність переходу
new_client = pd.DataFrame([{
    "age": 30,
    "tenure": 24,
    "usage": 120,
    "support": 0,
    "plan": "сімейний",
    "region": "захід"
}])

decision = pipe.predict(new_client)[0]
proba = pipe.predict_proba(new_client)[0, 1]

print(f"Рішення:      {'перейде на преміум' if decision == 1 else 'не перейде'}")
print(f"Ймовірність:  {proba:.2%}")

Рішення:      перейде на преміум
Ймовірність:  99.45%


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [34]:
from sklearn.model_selection import cross_val_score

# TODO: крос-валідація, виведи середнє ± розкид

scores = cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")

print("ROC-AUC по фолдах:")
for i, s in enumerate(scores, 1):
    print(f"  фолд {i}: {s:.3f}")

print()
print(f"Середнє:  {scores.mean():.3f}")
print(f"Розкид:   ±{scores.std():.3f}")

ROC-AUC по фолдах:
  фолд 1: 0.934
  фолд 2: 0.931
  фолд 3: 0.940
  фолд 4: 0.891
  фолд 5: 0.921

Середнє:  0.923
Розкид:   ±0.018


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [ ]:
# Місце для бонусних експериментів

---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?
2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?
3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?
4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?
5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.